# 04b — LLM skill extraction on the full corpus (shard 0 of 4)
Runs the NB4 extraction prompt (Qwen2.5-7B-Instruct) on every ad whose position in `hn_clean.csv` (sorted by
`comment_id`) is ≡ 0 mod 4. Engine: **vLLM** with the official AWQ 4-bit checkpoint; if vLLM can't
be installed or loaded, it falls back to transformers + bitsandbytes 4-bit (the NB4 setup), within a time budget.
Output: `llm_raw_full_shard0.csv` (`comment_id`, `raw`, `engine`), saved after every chunk.
NB4 reads all shards, re-scores the 400 labelled ads with these outputs, and uses them for the full corpus.

In [ ]:
import subprocess, sys
def pip(*pkgs):
    """Quiet pip install inside Kaggle (needs Settings -> Internet: On)."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    print("pip install", " ".join(pkgs), "->", "ok" if r.returncode == 0 else "FAILED\n" + r.stderr[-1500:])

import os, sys, time, json, glob
import numpy as np, pandas as pd
SHARD, N_SHARDS = 0, 4
OUT = "/kaggle/working"
OUT_FILE = f"{OUT}/llm_raw_full_shard{SHARD}.csv"
TIME_BUDGET_H = 11.0                     # Kaggle sessions stop at 12 h
T_START = time.time()
VLLM_OK = False
try:
    pip("vllm")
    import vllm  # noqa
    VLLM_OK = True
except Exception as e:
    print("vLLM not available:", str(e)[:200])

import glob, os
def find_file(name, prefer=None, required=True):
    """Find a file anywhere under /kaggle/input (Kaggle mounts datasets at different depths)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True), key=len)
    if prefer:
        hits = [h for h in hits if f"/{prefer}/" in h] or hits
    if not hits:
        if required:
            raise FileNotFoundError(f"'{name}' not found under /kaggle/input. "
                                    f"Add the dataset{' ' + repr(prefer) if prefer else ''} via 'Add Input'.")
        return None
    return hits[0]

def find_dir(dataset, marker, required=True):
    """Folder of input dataset `dataset`, identified by a file it must contain."""
    p = find_file(marker, prefer=dataset, required=required)
    return os.path.dirname(p) if p else None

hn = pd.read_csv(find_file("hn_clean.csv"), dtype={"comment_id": str}, usecols=["comment_id", "text"])
hn = hn[hn["text"].notna()].sort_values("comment_id").reset_index(drop=True)
mine = hn[hn.index % N_SHARDS == SHARD].copy()
mine["text"] = mine["text"].str[:2000]
print(f"shard {SHARD}/{N_SHARDS}: {len(mine):,} of {len(hn):,} ads")

In [ ]:
FEWSHOT = [
    ("Acme | Senior Backend Engineer | Remote (EU)\nWe build payment APIs in Go and PostgreSQL on AWS. "
     "You have 5+ years of backend experience, know Kubernetes, and write clear docs.",
     '["Go", "PostgreSQL", "AWS", "Kubernetes", "backend development", "technical writing"]'),
    ("Foodly | Marketing Manager | NYC\nOwn our email campaigns and SEO, analyse results in Google Analytics, "
     "and manage a small team.",
     '["email marketing", "SEO", "Google Analytics", "team management"]'),
]
INSTR = ("You extract skills from job ads. List every skill, tool, programming language, framework, method or "
         "area of knowledge that the ad asks candidates to have or will use in the job. Use short canonical names. "
         "Do not list benefits, locations, or the company's product. Answer with a JSON list of strings only.")

def chat(tok, user_text, examples=FEWSHOT, instr=INSTR):
    msgs = []
    for k, (q, a) in enumerate(examples):
        msgs += [{"role": "user", "content": (instr + "\n\nJob ad:\n" if k == 0 else "Job ad:\n") + q},
                 {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": ("" if examples else instr + "\n\n") + user_text})
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

done = pd.DataFrame(columns=["comment_id", "raw", "engine"])
if os.path.exists(OUT_FILE):
    done = pd.read_csv(OUT_FILE, dtype={"comment_id": str})
todo = mine[~mine["comment_id"].isin(done["comment_id"])]
print("to do:", len(todo))

In [ ]:
ENGINE = None
if VLLM_OK:
    try:
        from vllm import LLM, SamplingParams
        llm = LLM(model="Qwen/Qwen2.5-7B-Instruct-AWQ", quantization="awq", dtype="float16",
                  max_model_len=4096, gpu_memory_utilization=0.90, seed=42)
        tok = llm.get_tokenizer()
        sp = SamplingParams(temperature=0.0, max_tokens=200)
        ENGINE = "vllm:Qwen/Qwen2.5-7B-Instruct-AWQ"
        def gen(prompts):
            return [o.outputs[0].text for o in llm.generate(prompts, sp, use_tqdm=False)]
    except Exception as e:
        print("vLLM load failed:", str(e)[:300])
if ENGINE is None:
    pip("bitsandbytes>=0.43", "accelerate")
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    MID = "Qwen/Qwen2.5-7B-Instruct"
    tok = AutoTokenizer.from_pretrained(MID); tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MID, quantization_config=bnb, device_map="auto", dtype=torch.float16)
    ENGINE = "transformers-nf4:" + MID
    @torch.no_grad()
    def gen(prompts, bs=16):
        outs = []
        for i in range(0, len(prompts), bs):
            enc = tok(prompts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=3072,
                      add_special_tokens=False).to(model.device)
            g = model.generate(**enc, max_new_tokens=200, do_sample=False, pad_token_id=tok.pad_token_id)
            outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        return outs
print("engine:", ENGINE)

In [ ]:
CHUNK = 1000
for i in range(0, len(todo), CHUNK):
    if (time.time() - T_START) / 3600 > TIME_BUDGET_H:
        print("time budget reached; stopping (re-run this notebook to continue from the saved file)")
        break
    part = todo.iloc[i:i + CHUNK]
    t0 = time.time()
    raw = gen([chat(tok, "Job ad:\n" + t) for t in part["text"]])
    done = pd.concat([done, pd.DataFrame({"comment_id": part["comment_id"].values, "raw": raw, "engine": ENGINE})],
                     ignore_index=True)
    done.to_csv(OUT_FILE, index=False)
    print(f"{len(done):,}/{len(mine):,} ads | chunk {time.time() - t0:.0f}s | total {(time.time() - T_START) / 60:.0f} min")
print("finished:", len(done), "of", len(mine), "| empty answers:", (done["raw"].fillna("").str.strip() == "").sum())